Evan Edelstein
EN.605.645.82.SP26

# Module 9 - Programming Assignment

## Directions

1. Change the name of this file to be your JHED id as in `jsmith299.ipynb`. Because sure you use your JHED ID (it's made out of your name and not your student id which is just letters and numbers).
2. Make sure the notebook you submit is cleanly and fully executed. I do not grade unexecuted notebooks.
3. Submit your notebook back in Blackboard where you downloaded this file.

*Provide the output **exactly** as requested*

In [77]:
import json
import random
from copy import deepcopy
from math import inf, log2
from typing import Dict, List, NamedTuple, Tuple, Callable, Set

## Naive Bayes Classifier

For this assignment you will be implementing and evaluating a Naive Bayes Classifier with the same data from last week:

http://archive.ics.uci.edu/ml/datasets/Mushroom

(You should have downloaded it).

<div style="background: lemonchiffon; margin:20px; padding: 20px;">
    <strong>Important</strong>
    <p>
        No Pandas. The only acceptable libraries in this class are those contained in the `environment.yml`. No OOP, either. You can use Dicts, NamedTuples, Data Classes, etc. as your abstract data type (ADT).
    </p>
</div>


You'll first need to calculate all of the necessary probabilities using a `train` function. A flag will control whether or not you use "+1 Smoothing" or not. You'll then need to have a `classify` function that takes your probabilities, a List of instances (possibly a list of 1) and returns a List of Tuples. Each Tuple has the best class in the first position and a dict with a key for every possible class label and the associated *normalized* probability. For example, if we have given the `classify` function a list of 2 observations, we would get the following back:

```
[("e", {"e": 0.98, "p": 0.02}), ("p", {"e": 0.34, "p": 0.66})]
```

when calculating the error rate of your classifier, you should pick the class label with the highest probability; you can write a simple function that takes the Dict and returns that class label.

As a reminder, the Naive Bayes Classifier generates the *unnormalized* probabilities from the numerator of Bayes Rule:

$$P(C|A) \propto P(A|C)P(C)$$

where C is the class and A are the attributes (data). Since the normalizer of Bayes Rule is the *sum* of all possible numerators and you have to calculate them all, the normalizer is just the sum of the probabilities.

You will have the same basic functions as the last module's assignment and some of them can be reused or at least repurposed.

`train` takes training_data and returns a Naive Bayes Classifier (NBC) as a data structure. There are many options including namedtuples and just plain old nested dictionaries. **No OOP**.

```
def train(training_data, smoothing=True):
   # returns the "classifier" (however you decided to represent the probability tables).
```

The `smoothing` value defaults to True. You should handle both cases.

`classify` takes a NBC produced from the function above and applies it to labeled data (like the test set) or unlabeled data (like some new data). (This is not the same `classify` as the pseudocode which classifies only one instance at a time; it can call it though).

```
def classify(nbc, observations, labeled=True):
    # returns a list of tuples, the argmax and the raw data as per the pseudocode.
```

`evaluate` takes a data set with labels (like the training set or test set) and the classification result and calculates the classification error rate:

$$error\_rate=\frac{errors}{n}$$

Do not use anything else as evaluation metric or the submission will be deemed incomplete, ie, an "F". (Hint: accuracy rate is not the error rate!).

`cross_validate` takes the data and uses 10 fold cross validation (from Module 3!) to `train`, `classify`, and `evaluate`. **Remember to shuffle your data before you create your folds**. I leave the exact signature of `cross_validate` to you but you should write it so that you can use it with *any* `classify` function of the same form (using higher order functions and partial application). If you did so last time, you can reuse it for this assignment.

Following Module 3's material (course notes), `cross_validate` should print out a table in exactly the same format. What you are looking for here is a consistent evaluation metric cross the folds. Print the error rate to 4 decimal places. **Do not convert to a percentage.**


To summarize...

Apply the Naive Bayes Classifier algorithm to the Mushroom data set using 10 fold cross validation and the error rate as the evaluation metric. You will do this *twice*. Once with smoothing=True and once with smoothing=False. You should follow up with a brief hypothesis/explanation for the similarities or differences in the results. You may also compare the results to the Decision Tree and why you think they're different (if they are).

### Provided Functions

You do not need to document these.

You can use this function to read the data file.

In [78]:
def parse_data(file_name: str) -> list[list]:
    data = []
    file = open(file_name, "r")
    for line in file:
        datum = line.rstrip().split(",")
        data.append(datum)
    random.shuffle(data)
    return data

You can use this function to create 10 folds for 5x2 cross validation.

In [79]:
def create_folds(xs: list, n: int) -> list[list[list]]:
    k, m = divmod(len(xs), n)
    # be careful of generators...
    return list(xs[i * k + min(i, m):(i + 1) * k + min(i + 1, m)] for i in range(n))

Put your code after this line:

-----

# I/O and Data Parsing

<a id="parse_attributes"></a>
## parse_attributes

*`parse_attributes` parse an attributes json file given by filename. The file contains a mapping of each feature to a nested map of encoding of the attribute in the data, to the full name of the attribute to be displayed in the tree. Two dictionaries are returned. The first is a map of each feature to a tuple containing the index of that feature in the dataset and the list of attributes in the domain of the feature. The second maps each feature and encoded attribute to the full name of the attribute. Note, the label and its domain should be included in the json file. The order of each feature in the json should match the order they appear in each row of the data set.*

* **filename** str - filepath to a json of features and attributes - The order of each feature should match the order they appear in the data set.


**returns** Tuple[Dict[str, Tuple[int, List[str]]], Dict[str, Dict[str. str]]] - a map of each feature (and label) to a tuple with its position in the dataset and a list of attributes in the domain of the feature, and a nested map of each feature and encoded attribute to the full name of the attribute

In [80]:
def parse_attributes(filename: str) -> Tuple[Dict[str, Tuple[int, List[str]]], Dict[str, Dict[str, str]]]:
    abrv2fullname: Dict[str, Dict[str, str]] = {}
    attributes: Dict[str, Tuple[int, List[str]]] = {}

    with open(filename, "r") as fh:
        data: Dict[str, Dict[str, str]] = json.load(fh)

    for idx, (feature, attrs) in enumerate(data.items()):
        abrv2fullname[feature] = {}
        attributes[feature] = (idx, [])
        for name, code in attrs.items():
            attributes[feature][1].append(name)
            abrv2fullname[feature][code] = name

    return attributes, abrv2fullname

In [81]:
filename = "./agaricus-lepiota-3.attrs.json"
attributes, abrv2fullname = parse_attributes(filename)

attribute_keys = [
    "mushroom-type",
    "cap-shape",
    "cap-surface",
    "cap-color",
    "bruises?",
    "odor",
    "gill-attachment",
    "gill-spacing",
    "gill-size",
    "gill-color",
    "stalk-shape",
    "stalk-root",
    "stalk-surface-above-ring",
    "stalk-surface-below-ring",
    "stalk-color-above-ring",
    "stalk-color-below-ring",
    "veil-type",
    "veil-color",
    "ring-number",
    "ring-type",
    "spore-print-color",
    "population",
    "habitat",
]

assert list(attributes.keys()) == attribute_keys  # test 1 - all keys are present
assert all([len(a) > 1 for a in attr] for _, attr in attributes.values())  # test 2 - all attribute names are full name
assert all([len(k) == 1 and len(v) > 0 for k, v in a.items()] for a in abrv2fullname.values())  # test 3 - can map from single letter to full name

<a id="rename_data"></a>
## rename_data

*`rename_data` Convert all values in data from an encoded attribute to the full attribute name given by the nested dictionary abrv2name, which maps each feature to a dictionary of encoded attribute values to full attribute name. A dictionary that maps each feature to its domain is also required. If a value in data cannot be translated, None is returned.*

* **data** List[List[str]] - a 2d list of observed attributes.
* **attributes** Dict[str, Tuple[int, List[str]]] - a map of each feature (and label) to a tuple of its position in the dataset and a list of attributes in the domain of the feature.
* **abrv2name** Dict[str, Dict[str. str]] - a nested map of each feature and encoded attribute to the full name of the attribute


**returns** List[List[str]] | None - a copy of data with all values translated to their full name or None if a value cannot be translated

In [82]:
def rename_data(data: List[List[str]], attributes: Dict[str, Tuple[int, List[str]]], abrv2name: Dict[str, Dict[str, str]]) -> List[List[str]] | None:
    new_data = []
    for row in data:
        if len(row) != len(attributes):
            return None

        new_row = []
        for value, attr in zip(row, attributes):
            if attr in abrv2name and value in abrv2name[attr]:
                new_row.append(abrv2name[attr][value])
            else:
                return None
        new_data.append(new_row)

    return new_data

In [83]:
data = [["a", "b", "c"], ["a", "b", "c"]]
attributes = {"1": (0, ["a"]), "2": (1, ["b"]), "3": (2, ["c"])}
abrv2name = {"1": {"a": "aaa"}, "2": {"b": "bbb"}, "3": {"c": "ccc"}}

result = rename_data(data, attributes, abrv2name)
assert result is not None and result[0][0] == "aaa" and result[0][1] == "bbb" and result[0][2] == "ccc" and result[1][0] == "aaa" and result[1][1] == "bbb" and result[1][2] == "ccc"  # test 1 - normal replacement


data = [["a", "b", "c"]]
attributes = {"1": (0, ["a"]), "2": (1, ["b"]), "3": (2, ["c"])}
abrv2name = {"2": {"b": "bbb"}, "3": {"c": "ccc"}}

result = rename_data(data, attributes, abrv2name)
assert result is None  # test 2 - missing attribute in map


data = [["a", "b", "c", "d"]]
attributes = {"1": (0, ["a"]), "2": (1, ["b"]), "3": (2, ["c"])}
abrv2name = {"1": {"a": "aaa"}, "2": {"b": "bbb"}, "3": {"c": "ccc"}}
result = rename_data(data, attributes, abrv2name)
assert result is None  # test 3 - extra value in data

<a id="split_features"></a>
## split_features

*`split_features` Given a mapping where each key is a feature, extract all the keys except the one matching label.* **Used by**: [run_model](#run_model)

* **attributes** Dict[str, Tuple[int, List[str]]] - a map of each feature (or label) to a tuple of its position in the dataset and a list of attributes in the domain of the feature.
* **label** str - key to skip when scanning attributes


**returns** Set[str] - a set of all non-label features 

In [84]:
def split_features(attributes: Dict[str, Tuple[int, List[str]]], label: str) -> List[str]:
    return [i for i in attributes if i != label]

In [85]:
attributes = {"1": (0, ["a"]), "2": (1, ["b"]), "3": (2, ["c"])}
label = "3"
features = split_features(attributes, label)
assert features == ["1", "2"]  # test 1 - splits features and labels

attributes = {"1": (0, ["a"]), "2": (1, ["b"]), "3": (2, ["c"])}
label = "4"
features = split_features(attributes, label)
assert features == ["1", "2", "3"]  # test 2 - label doesn't exist in attributes


attributes = {"1": (0, ["a"])}
label = "1"
features = split_features(attributes, label)
assert features == []  # test 3 - only label so features is empty

# Naive Bayes Classifier

In [86]:
NBC = NamedTuple("NBC", [("pc", Dict[str,float]), ("pf", Dict[str, Dict[str, Dict[str, float]]])])

In [87]:
def naive_bayes_classifier(data: List[List[str]], features: List[str], attributes: Dict[str, Tuple[int, List[str]]], label: str, trace: bool = False) -> NBC:
    pf = {}
    pcs = {}
    label_idx, labels = attributes[label]
    for label in labels:
        label_rows = [row for row in data if row[label_idx] == label]
        pc = len(label_rows) / len(data)
        pf[label] = {}
        pcs[label] = pc
        # for feature, (feature_idx, domain) in attributes.items():
        for feature in features:
            feature_idx, domain = attributes[feature]
            pf[label][feature] = {}
            for attr in domain:
                fi = [row for row in label_rows if row[feature_idx] == attr]
                pf[label][feature][attr] = (len(fi) + 1) / (len(label_rows) + 1)
    model = NBC(pcs, pf)
    return model

# Model

<a id="train"></a>
## train

*`train` train decision tree using id3 algorithm on data.* **Uses** [get_majority_label](#get_majority_label) and [id3](#id3)

* **data** List[List[str]] - a 2d list of observed attributes and label
* **features** Set[str] - set of features
* **attributes** Dict[str, Tuple[int, List[str]]] - a map of each feature (or label) to a tuple of its position in the dataset and a list of attributes in the domain of the feature.
* **label** str - label name
* **trace** bool - if True print debug information

**returns** Node - root of decision tree

In [88]:
def train(data: List[List[str]], features: List[str], attributes: Dict[str, Tuple[int, List[str]]], label: str, trace=False) -> NBC:
    return naive_bayes_classifier(data, features, attributes, label, trace)


<a id="classify"></a>
## classify

*`classify` Given a decision tree and a 2d list of observed attributes, estimate the label of each observation.* **Uses** [traverse_tree](#traverse_tree) 

* **tree** Node - root node of trained decision tree
* **observations** List[List[str]] - a 2d list of observed attributes
* **feature_indices** Dict[str, int] - mapping of feature to column index in data

**returns** List[str] - list of estimated labels for each row in observations

In [89]:
def classify(model: NBC, observations: List[List[str]], features: List[str], labels: List[str]) -> List[Tuple[str, Dict[str, float]]]:
    classifications = []
    for row in observations:
        estimates = {}
        for label in labels:
            c = model.pc[label]
            for feature, attr in zip(features, row):
                c *= model.pf[label][feature][attr]
            estimates[label] = c
        estimate_label = max(estimates.items(), key=lambda x: x[1])[0]
        classifications.append((estimate_label, estimates)) 
    return classifications

<a id="evaluate"></a>
## evaluate

*`evaluate` Given a list of true labels and a list of estimated labels, count the number of errors and create a confusion matrix by calculating the number TP, TN, FP, FN.* **Uses** [traverse_tree](#traverse_tree) 

* **truth_set** List[str] - list of true labels
* **classifications** List[str] - list of estimated labels


**returns** Tuplep[int,Dict[str, int]]   - number of non-matching estimates, confusion matrix

In [90]:
def evaluate(truth_set: List[str], classifications: List[Tuple[str, Dict[str, float]]], labels: List[str]) -> Tuple[int, Dict[str, int]]:
    errors = 0
    fold_cm: Dict[str, int] = {"TN": 0, "TP": 0, "FN": 0, "FP": 0}
    for true_label, estimate_prob in zip(truth_set, classifications):
        estimate = estimate_prob[0] # highest probability label 
        if true_label == estimate:
            if estimate == labels[1]:
                fold_cm["TP"] += 1
            else:
                fold_cm["TN"] += 1

        elif true_label != estimate:
            errors += 1
            if true_label == labels[0]:
                fold_cm["FP"] += 1
            else:
                fold_cm["FN"] += 1
    return errors, fold_cm

<a id="divide_folds"></a>
## divide_folds

*`divide_folds` Shuffle a dataset and divide into n leave-one-out training and test pairs.* **Uses** [create_folds](#create_folds) 

* **data** List[List[str]] - 2d list of observations and label
* **n_folds** int - number of folds to generate

**returns** List[Tuple[List[List[str]], List[List[str]]]] - List of tuple containing a training set and test set for each fold

In [91]:
def divide_folds(data: List[List[str]], n_folds: int = 10) -> List[Tuple[List[List[str]], List[List[str]]]]:
    random.shuffle(data)
    folds = create_folds(data, n_folds)

    k_folds = []
    for idx, test_fold in enumerate(folds):
        training_set = []
        for idx2, train_fold in enumerate(folds):
            if idx == idx2:
                continue
            training_set.extend(train_fold)

        k_folds.append((training_set, test_fold))
    return k_folds

<a id="cross_validate"></a>
## cross_validate

*`cross_validate` perform n_fold cross validation. For each fold, a train_fn is used to produce a model from the training data in the fold. The model is used to classify the test data in the fold using classify_fn. The classification is evaluated using the evaluate_fn. The error rate and a confusion matrix built from all the folds is returned.* 

* **data** List[List[str]] - a 2d list of observed attributes and label
* **features** Set[str] - set of features
* **attributes** Dict[str, Tuple[int, List[str]]] - a map of each feature (or label) to a tuple of its position in the dataset and a list of attributes in the domain of the feature.
* **label** str - label name
* **train_fn** Callable - function to produce model from training data
* **classify_fn** Callable - function to generate label estimates on test data using a model
* **evaluate_fn** Callable - function to collect the number of errors from the classification, as well as, update a confusion matrix.
* **n_folds** int - number of folds to use
* **trace** bool - if True print debug information

**returns** Tuple[float, Dict[str, int]] -  error rate and confusion matrix from all folds

In [ ]:
def cross_validate(
    data: List[List[str]], features: List[str], attributes: Dict[str, Tuple[int, List[str]]], label: str, train_fn: Callable = train, classify_fn: Callable = classify, eval_fn: Callable = evaluate, n_folds: int = 10, trace: bool = False
) -> Tuple[float, Dict[str, int]]:
    confusion_matrices: List[Dict[str, int]] = []
    total_errors = []

    label_idx, label_values = attributes[label]

    for k, (training_set, test_set) in enumerate(divide_folds(data, n_folds)):
        model = train_fn(training_set, features, attributes, label, trace)
        masked_test_set = [[i for c,i in enumerate(row) if c!=label_idx ] for row in test_set]
        classifications = classify_fn(model, masked_test_set, features, label_values)
        truth_set = [row[label_idx] for row in test_set]
        errors, cm = eval_fn(truth_set, classifications, label_values)
        error_rate = errors / len(test_set)
        total_errors.append(error_rate)
        confusion_matrices.append(cm)
        print(f"Fold {k}\nTraining size: {len(training_set)} | Test size: {len(test_set)}\nError rate: {error_rate}\nConfusion matrix:\nTP={cm['TP']}  FP={cm['FP']}\nFN={cm['FN']}  TN={cm['TN']}\n")
    total_error_rate = sum(total_errors) / len(total_errors)
    return total_error_rate, {k: sum([cm[k] for cm in confusion_matrices]) for k in confusion_matrices[0]}

# Run

<a id="run_model"></a>
## run_model

*`run_model` perform 10-fold cross validation on a dataset and then print the decision tree trained on the entire dataset.* **Uses** [split_features](#split_features), [cross_validate](#cross_validate), [train](#train) and [pretty_print_tree](#pretty_print_tree)

* **data** List[List[str]] - a 2d list of observed attributes and label
* **features** Set[str] - set of features
* **attributes** Dict[str, Tuple[int, List[str]]] - a map of each feature (or label) to a tuple of its position in the dataset and a list of attributes in the domain of the feature.
* **label** str - label name
* **trace** bool - if True print debug information

**returns** 

In [93]:
def run_model(data: List[List[str]], attributes: Dict[str, Tuple[int, List[str]]], label: str, trace: bool = False):
    n_folds = 10

    features = split_features(attributes, label)
    
    avrg_error_rate, cm = cross_validate(data, features, attributes, label, n_folds=n_folds)
    print(f"\nTotal Confusion Matrix ({n_folds}-fold CV):")
    print(f"TP={cm['TP']}  FP={cm['FP']}\nFN={cm['FN']}  TN={cm['TN']}")
    print(f"\nAverage Error Rate ({n_folds}-fold CV): {avrg_error_rate}")

    model = train(data, features, attributes, label, trace=trace)
    assert model is not None

    print("\nNBC model:")
    # prrety_print_nbc(model)

In [94]:
attributes = {"Shape": (0, ["round", "square"]), "Size": (1, ["large", "small"]), "Color": (2, ["blue", "green", "red"]), "Safe?": (3, ["yes", "no"])}
label = "Safe?"

data = [
    ["round", "large", "blue", "no"],
    ["square", "large", "green", "yes"],
    ["square", "small", "red", "no"],
    ["round", "large", "red", "yes"],
    ["square", "small", "blue", "no"],
    ["round", "small", "blue", "no"],
    ["round", "small", "red", "yes"],
    ["square", "small", "green", "no"],
    ["round", "large", "green", "yes"],
    ["square", "large", "green", "yes"],
    ["square", "large", "red", "no"],
    ["square", "large", "green", "yes"],
    ["round", "large", "red", "yes"],
    ["square", "small", "red", "no"],
    ["round", "small", "green", "no"],
]

run_model(data, attributes, label)

Fold 0
Training size: 13 | Test size: 2
Error rate: 0.0
Confusion matrix:
TP=1  FP=0
FN=0  TN=1

Fold 1
Training size: 13 | Test size: 2
Error rate: 0.0
Confusion matrix:
TP=1  FP=0
FN=0  TN=1

Fold 2
Training size: 13 | Test size: 2
Error rate: 0.5
Confusion matrix:
TP=0  FP=0
FN=1  TN=1

Fold 3
Training size: 13 | Test size: 2
Error rate: 0.0
Confusion matrix:
TP=0  FP=0
FN=0  TN=2

Fold 4
Training size: 13 | Test size: 2
Error rate: 0.5
Confusion matrix:
TP=1  FP=0
FN=1  TN=0

Fold 5
Training size: 14 | Test size: 1
Error rate: 1.0
Confusion matrix:
TP=0  FP=0
FN=1  TN=0

Fold 6
Training size: 14 | Test size: 1
Error rate: 0.0
Confusion matrix:
TP=0  FP=0
FN=0  TN=1

Fold 7
Training size: 14 | Test size: 1
Error rate: 0.0
Confusion matrix:
TP=1  FP=0
FN=0  TN=0

Fold 8
Training size: 14 | Test size: 1
Error rate: 1.0
Confusion matrix:
TP=0  FP=1
FN=0  TN=0

Fold 9
Training size: 14 | Test size: 1
Error rate: 0.0
Confusion matrix:
TP=1  FP=0
FN=0  TN=0

[0.0, 0.0, 0.5, 0.0, 0.5, 1.0,

In [95]:
trace = False
data = parse_data("./agaricus-lepiota-1-2.data")
attributes, abrv2name = parse_attributes("./agaricus-lepiota-3.attrs.json")
label = "mushroom-type"

data = rename_data(data, attributes, abrv2name)
assert data is not None

run_model(data, attributes, label)

Fold 0
Training size: 7311 | Test size: 813
Error rate: 0.03567035670356704
Confusion matrix:
TP=418  FP=29
FN=0  TN=366

Fold 1
Training size: 7311 | Test size: 813
Error rate: 0.03690036900369004
Confusion matrix:
TP=424  FP=27
FN=3  TN=359

Fold 2
Training size: 7311 | Test size: 813
Error rate: 0.04059040590405904
Confusion matrix:
TP=412  FP=32
FN=1  TN=368

Fold 3
Training size: 7311 | Test size: 813
Error rate: 0.04059040590405904
Confusion matrix:
TP=418  FP=30
FN=3  TN=362

Fold 4
Training size: 7312 | Test size: 812
Error rate: 0.05172413793103448
Confusion matrix:
TP=435  FP=42
FN=0  TN=335

Fold 5
Training size: 7312 | Test size: 812
Error rate: 0.0603448275862069
Confusion matrix:
TP=399  FP=47
FN=2  TN=364

Fold 6
Training size: 7312 | Test size: 812
Error rate: 0.05295566502463054
Confusion matrix:
TP=412  FP=38
FN=5  TN=357

Fold 7
Training size: 7312 | Test size: 812
Error rate: 0.04064039408866995
Confusion matrix:
TP=425  FP=32
FN=1  TN=354

Fold 8
Training size: 731

## Before You Submit...

1. Did you provide output exactly as requested?
2. Did you re-execute the entire notebook? ("Restart Kernel and Rull All Cells...")
3. If you did not complete the assignment or had difficulty please explain what gave you the most difficulty in the Markdown cell below.
4. Did you change the name of the file to `jhed_id.ipynb`?

Do not submit any other files.